# PAMPAr-V3 Ablation Monitor

Real-time monitoring de los 4 experimentos de ablación.

| Experimento | Descripción |
|---|---|
| `pampar_v3` | PAMPAr-V3 completo (control) |
| `no_llaves` | Sin LLAVES — solo routing aprendido |
| `single_stream` | 1 stream — sin estructura 2D |
| `vanilla_gpt` | GPT estándar ~62M params |

In [ ]:
import json
import time
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib
from IPython.display import clear_output, display

matplotlib.rcParams.update({
    "figure.figsize": (14, 8),
    "font.size": 12,
    "axes.grid": True,
    "grid.alpha": 0.3,
})

# Ajustar según dónde se ejecute
RESULTS_DIR = Path("/workspace/PAMPAr-Coder/ablation_results")
# Si es local:
# RESULTS_DIR = Path("../ablation_results")

EXPERIMENTS = ["pampar_v3", "no_llaves", "single_stream", "vanilla_gpt"]
COLORS = {"pampar_v3": "#2ecc71", "no_llaves": "#e74c3c", "single_stream": "#3498db", "vanilla_gpt": "#9b59b6"}
LABELS = {
    "pampar_v3": "PAMPAr-V3 (control)",
    "no_llaves": "Sin LLAVES",
    "single_stream": "1 Stream",
    "vanilla_gpt": "Vanilla GPT",
}

print(f"Results dir: {RESULTS_DIR}")
print(f"Exists: {RESULTS_DIR.exists()}")

In [ ]:
def load_metrics(experiment: str) -> dict:
    """Carga métricas de un experimento."""
    metrics_path = RESULTS_DIR / experiment / "metrics.jsonl"
    if not metrics_path.exists():
        return {"steps": [], "losses": [], "avg_losses": [], "ppls": [],
                "eval_steps": [], "eval_losses": [], "eval_ppls": [], "speeds": []}

    steps, losses, avg_losses, ppls, speeds = [], [], [], [], []
    eval_steps, eval_losses, eval_ppls = [], [], []

    with metrics_path.open(encoding="utf-8") as f:
        for line in f:
            r = json.loads(line.strip())
            if r.get("type") == "eval":
                eval_steps.append(r["step"])
                eval_losses.append(r["eval_loss"])
                eval_ppls.append(r["eval_ppl"])
            else:
                steps.append(r["step"])
                losses.append(r["loss"])
                avg_losses.append(r["avg_loss"])
                ppls.append(r["ppl"])
                speeds.append(r.get("steps_per_sec", 0))

    return {
        "steps": steps, "losses": losses, "avg_losses": avg_losses,
        "ppls": ppls, "speeds": speeds,
        "eval_steps": eval_steps, "eval_losses": eval_losses, "eval_ppls": eval_ppls,
    }

In [ ]:
def plot_ablation(auto_refresh: bool = False) -> None:
    """Genera los gráficos de ablación."""
    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    fig.suptitle("PAMPAr-V3 Ablation Study", fontsize=16, fontweight="bold")

    all_data = {exp: load_metrics(exp) for exp in EXPERIMENTS}

    # ── 1. Training Loss (avg100) ────────────────────────────────────────
    ax = axes[0, 0]
    ax.set_title("Training Loss (avg 100)")
    for exp in EXPERIMENTS:
        d = all_data[exp]
        if d["steps"]:
            ax.plot(d["steps"], d["avg_losses"], color=COLORS[exp],
                    label=f"{LABELS[exp]} ({d['avg_losses'][-1]:.3f})", alpha=0.85)
    ax.set_xlabel("Steps")
    ax.set_ylabel("Loss")
    ax.legend(fontsize=9)
    ax.set_ylim(bottom=0)

    # ── 2. Eval Loss ─────────────────────────────────────────────────────
    ax = axes[0, 1]
    ax.set_title("Eval Loss (cada 2K steps)")
    for exp in EXPERIMENTS:
        d = all_data[exp]
        if d["eval_steps"]:
            ax.plot(d["eval_steps"], d["eval_losses"], color=COLORS[exp],
                    marker="o", markersize=5, label=LABELS[exp], alpha=0.85)
    ax.set_xlabel("Steps")
    ax.set_ylabel("Eval Loss")
    ax.legend(fontsize=9)
    ax.set_ylim(bottom=0)

    # ── 3. Perplexity ────────────────────────────────────────────────────
    ax = axes[1, 0]
    ax.set_title("Perplexity (train avg)")
    for exp in EXPERIMENTS:
        d = all_data[exp]
        if d["steps"]:
            ax.plot(d["steps"], d["ppls"], color=COLORS[exp],
                    label=LABELS[exp], alpha=0.85)
    ax.set_xlabel("Steps")
    ax.set_ylabel("Perplexity")
    ax.set_yscale("log")
    ax.legend(fontsize=9)

    # ── 4. Training Speed ────────────────────────────────────────────────
    ax = axes[1, 1]
    ax.set_title("Training Speed")
    for exp in EXPERIMENTS:
        d = all_data[exp]
        if d["steps"] and d["speeds"]:
            ax.plot(d["steps"], d["speeds"], color=COLORS[exp],
                    label=LABELS[exp], alpha=0.7)
    ax.set_xlabel("Steps")
    ax.set_ylabel("Steps/sec")
    ax.legend(fontsize=9)

    plt.tight_layout()
    plt.show()

    # ── Summary table ────────────────────────────────────────────────────
    print("\n" + "═" * 72)
    print(f"{'Experiment':<20} {'Steps':>8} {'Loss':>8} {'Eval Loss':>10} {'PPL':>8} {'Speed':>10}")
    print("─" * 72)
    for exp in EXPERIMENTS:
        d = all_data[exp]
        step = d["steps"][-1] if d["steps"] else 0
        loss = d["avg_losses"][-1] if d["avg_losses"] else 0
        ev = d["eval_losses"][-1] if d["eval_losses"] else 0
        ppl = d["ppls"][-1] if d["ppls"] else 0
        spd = d["speeds"][-1] if d["speeds"] else 0
        print(f"{LABELS[exp]:<20} {step:>8} {loss:>8.3f} {ev:>10.3f} {ppl:>8.1f} {spd:>8.1f} s/s")
    print("═" * 72)

## Ejecutar una vez

In [ ]:
plot_ablation()

## Auto-refresh (ejecutar y dejar corriendo)

In [ ]:
REFRESH_INTERVAL = 60  # segundos

try:
    while True:
        clear_output(wait=True)
        print(f"Última actualización: {time.strftime('%H:%M:%S')}")
        print(f"Auto-refresh cada {REFRESH_INTERVAL}s (Interrupt kernel para parar)\n")
        plot_ablation()
        time.sleep(REFRESH_INTERVAL)
except KeyboardInterrupt:
    print("\nMonitoreo detenido.")

## Análisis final (después de completar los 4 experimentos)

In [ ]:
def final_analysis() -> None:
    """Análisis comparativo final de la ablación."""
    all_data = {exp: load_metrics(exp) for exp in EXPERIMENTS}
    control = all_data["pampar_v3"]

    if not control["eval_losses"]:
        print("No hay datos de eval para el control aún.")
        return

    control_final = control["eval_losses"][-1]

    print("\n" + "═" * 72)
    print("ABLATION STUDY — FINAL RESULTS")
    print("═" * 72)
    print(f"\nControl (PAMPAr-V3): eval_loss = {control_final:.4f}\n")
    print(f"{'Experiment':<20} {'Eval Loss':>10} {'Δ Loss':>10} {'Δ %':>8} {'Verdict':>12}")
    print("─" * 72)

    for exp in EXPERIMENTS:
        d = all_data[exp]
        if not d["eval_losses"]:
            print(f"{LABELS[exp]:<20} {'N/A':>10}")
            continue
        ev = d["eval_losses"][-1]
        delta = ev - control_final
        pct = (delta / control_final) * 100 if control_final > 0 else 0
        verdict = "CONTROL" if exp == "pampar_v3" else ("WORSE ↑" if delta > 0.01 else "SIMILAR ≈" if abs(delta) <= 0.01 else "BETTER ↓")
        print(f"{LABELS[exp]:<20} {ev:>10.4f} {delta:>+10.4f} {pct:>+7.1f}% {verdict:>12}")

    print("═" * 72)
    print("\nInterpretación:")
    print("  - Si no_llaves es WORSE → LLAVES aporta valor (routing por reglas ayuda)")
    print("  - Si single_stream es WORSE → La estructura 2D multi-stream ayuda")
    print("  - Si vanilla_gpt es WORSE → La arquitectura completa de PAMPAr supera a un GPT básico")
    print("  - Si vanilla_gpt es SIMILAR/BETTER → PAMPAr no justifica su complejidad")

final_analysis()